# MiroFish 本機路徑基準測試（Colab）

在 Colab 的 GPU 上跑 A/B 基準（`ab_suite.py`／`ab_pool.py`），結果存到 Google Drive。所有模型都在這台 runtime 上執行，不呼叫外部 API。

**使用方式**
1. 執行階段 → 變更執行階段類型 → 選 GPU（建議先用 **L4**；A100 較快，但運算單元耗得較多）。
2. 修改下方「參數」一格，再從上往下執行。
3. 中斷後重新執行即可續跑：已完成的 run 會從 Drive 複製回來並沿用。

**注意**：Colab 上的模型伺服器（vLLM、BF16）和本機的 llama.cpp（GGUF）不同。同一個比較的 A、B 兩組都要在這裡跑，數字不要和本機的結果混用。

In [ ]:
# ===== 參數 =====
BRANCH = "EndeavorYen/53-scan-reliability"   # 要測的分支
EXPERIMENT = "suite"            # "suite" = 評估庫 6 個情境；"calibration" = calibration 庫（seeds 11–13）
RUN_NAME = "suite-vllm-bf16"    # Drive 上的結果資料夾名稱
SEEDS = [1, 2, 3, 4, 5]
ROUNDS = 24
JOBS = 16                       # 同時跑幾個模擬；記憶體不夠時會自動等待
MODEL = "Qwen/Qwen3.5-4B"       # BF16 原版；要和本機 Docker 一致可改 "SubSir/Qwen3.5-4B-AWQ"
DRIVE_DIR = "/content/drive/MyDrive/mirofish-runs"
PROBE_ONLY = False              # True：只跑吞吐量量測，估算額度後停止

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, time, json, pathlib
RUN_DIR = f"/content/runs/{RUN_NAME}"
DRIVE_RUN = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(DRIVE_RUN, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2; nproc

## 1. 安裝

In [ ]:
%%bash -s "$BRANCH"
set -e
[ -d /content/MiroFish ] || git clone -q --depth 1 -b "$1" https://github.com/EndeavorYen/MiroFish.git /content/MiroFish
cd /content/MiroFish && git log --oneline -1
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
cd backend && uv sync -q && echo "backend ready"
# vLLM 放在獨立環境，避免和後端的依賴衝突
[ -x /content/vllm-env/bin/vllm ] || (uv venv -q /content/vllm-env --python 3.12 && uv pip install -q --python /content/vllm-env vllm)
/content/vllm-env/bin/vllm --version
command -v rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync >/dev/null)

## 2. 啟動模型服務

- 生成與 System One 讀出：vLLM，port 8000。參數和 `docker-compose.yml` 相同：開啟 tool call 解析、關閉 Qwen3.5 的思考輸出、開啟 prefix cache。
- 向量：`scripts/embed_server.py`（transformers，斷詞和原始模型一致，見 #61），port 8001。

In [ ]:
BACKEND = "/content/MiroFish/backend"
UV = os.path.expanduser("~/.local/bin/uv")
logs = {}
def start(name, cmd, cwd=None):
    logs[name] = open(f"/content/{name}.log", "w")
    return subprocess.Popen(cmd, stdout=logs[name], stderr=subprocess.STDOUT, cwd=cwd)

vllm = start("vllm", ["/content/vllm-env/bin/vllm", "serve", MODEL, "--served-model-name", "qwen3.5-4b",
                      "--host", "127.0.0.1", "--port", "8000", "--max-model-len", "8192",
                      "--gpu-memory-utilization", "0.85", "--max-num-seqs", "128", "--enable-prefix-caching",
                      "--max-logprobs", "20", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                      "--default-chat-template-kwargs", '{"enable_thinking": false}'])
embed = start("embed", [UV, "run", "python", "scripts/embed_server.py", "--port", "8001"], cwd=BACKEND)

import urllib.request
def wait(url, name, timeout=1200):
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(url, timeout=3); print(name, "ready", round(time.time() - t), "s"); return
        except Exception:
            time.sleep(10)
    raise RuntimeError(f"{name} did not start; see /content/{name}.log")
wait("http://127.0.0.1:8001/health", "embed")
wait("http://127.0.0.1:8000/v1/models", "vllm")

## 3. 自我檢查（讀出、tool call、沒有思考輸出）

In [ ]:
import urllib.request
def post(path, payload):
    req = urllib.request.Request("http://127.0.0.1:8000" + path, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=120))

tool = post("/v1/chat/completions", {"model": "qwen3.5-4b", "messages": [{"role": "user", "content": "Like post 3."}],
    "tools": [{"type": "function", "function": {"name": "like_post", "parameters": {"type": "object",
    "properties": {"post_id": {"type": "integer"}}, "required": ["post_id"]}}}], "tool_choice": "auto", "max_tokens": 200})
msg = tool["choices"][0]["message"]
assert msg.get("tool_calls"), msg
assert "Thinking" not in (msg.get("content") or ""), msg["content"]
print("tool call OK:", msg["tool_calls"][0]["function"])

In [ ]:
%cd /content/MiroFish/backend
!{UV} run python scripts/calibrate_model.py --base-url http://127.0.0.1:8000/v1 --model qwen3.5-4b --out /content/readout_check.json > /dev/null
r = json.load(open("/content/readout_check.json"))
print(r["accuracy_by_category"], "positivity", r["positivity_bias"], "engage", r["engage_bias"], "latency ms", r["latency_ms_mean"])
assert r["accuracy_by_category"]["post_stance"] >= 0.8, "讀出準確度過低，先檢查模型與 prompt 格式"

## 4. 準備情境並量吞吐量

先準備 A（LLM 準備）與 B（模板準備），再用 golden 的 B 組跑 3 次 8 回合，用來估算整批的時間。

In [ ]:
%cd /content/MiroFish/backend
manifest = "tests/fixtures/scenarios/suite.json" if EXPERIMENT == "suite" else "tests/fixtures/calibration/calibration.json"
!rsync -a "{DRIVE_RUN}/" "{RUN_DIR}/" 2>/dev/null || true
# 只準備；已準備的目錄會沿用
!{UV} run python scripts/ab_suite.py --manifest {manifest} --out {RUN_DIR} --prepare-only > /content/prepare.log 2>&1 || tail -20 /content/prepare.log
!rsync -a "{RUN_DIR}/" "{DRIVE_RUN}/"
first = sorted(p for p in pathlib.Path(RUN_DIR).iterdir() if (p / "B" / "prepared.json").exists())[0]
t = time.time()
!{UV} run python scripts/ab_eval.py --work-a {first}/A --work-b {first}/B --out /content/probe --seeds 901 902 903 --rounds 8 --simulate-only B --jobs 3 > /content/probe.log 2>&1
probe = time.time() - t
scenarios = len([p for p in pathlib.Path(RUN_DIR).iterdir() if (p / "B" / "prepared.json").exists()])
runs = scenarios * 2 * len(SEEDS)
print(f"3 次 8 回合的 B run：{probe:.0f} 秒")
print(f"粗估：{runs} 次 {ROUNDS} 回合的模擬（A 組較慢），約 {probe / 3 * ROUNDS / 8 * runs / max(1, JOBS) * 1.5 / 3600:.1f} 小時")

## 5. 跑基準（可中斷、可續跑）

In [ ]:
if PROBE_ONLY:
    raise SystemExit("PROBE_ONLY=True：只做到量測")
import threading
stop = threading.Event()
def sync_loop():
    while not stop.wait(600):
        subprocess.run(["rsync", "-a", f"{RUN_DIR}/", f"{DRIVE_RUN}/"])
threading.Thread(target=sync_loop, daemon=True).start()
%cd /content/MiroFish/backend
!AB_MIN_FREE_GB=4 {UV} run python scripts/ab_suite.py --manifest {manifest} --out {RUN_DIR} --seeds {" ".join(map(str, SEEDS))} --rounds {ROUNDS} --jobs {JOBS} 2>&1 | grep -v INFO | tail -30
stop.set()
!rsync -a "{RUN_DIR}/" "{DRIVE_RUN}/"

## 6. 結果（G5 掃描閘門）

In [ ]:
%cd /content/MiroFish/backend
!{UV} run python scripts/scan_gates.py {RUN_DIR} --out {RUN_DIR}/scan_report.json
!rsync -a "{RUN_DIR}/" "{DRIVE_RUN}/"
print("結果在 Google Drive：", DRIVE_RUN)